text - 1차시
전처리 => 소문자 변환 / 토큰화 / 불용어 제거
정수 인코딩 => 단어 사전 구축 및 정수 인코딩

In [39]:
import os
import re
import pandas as pd
import pickle
from collections import Counter

In [40]:
# 불용어
stop_words = {
    # 대명사
    "i", "me", "my", "myself", "we", "our", "ourselves", 
    "you", "your", "yours", "yourself", "he", "him", "his", 
    "she", "her", "it", "its", "they", "them", "themselves",
    # be동사 / 조동사
    "is", "am", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did",
    "will", "would", "shall", "should", "can", "could",
    # 관사 / 전치사
    "a", "an", "the",
    "in", "on", "at", "to", "for", "of", "with", "by", "from",
    # 접속사 (but 제외)
    "and", "or", "if", "then",
    # 기타
    "this", "that", "these", "those",
    "there", "here", "what", "which", "who",
    "all", "each", "some", "any",
    "about", "up", "out",
}


In [41]:
#감정 단어 정의
emotion_words = {
    # positive
    "good", "great", "awesome", "amazing", "excellent", "fantastic",
    "wonderful", "nice", "love", "loved", "like", "liked",
    "fun", "funny", "cool", "best", "favorite", "enjoy", "enjoyed",
    "beautiful", "perfect", "strong", "interesting",

    # negative
    "bad", "terrible", "awful", "horrible", "worst", "boring",
    "hate", "hated", "dislike", "ugly", "annoying", "disappointing",
    "disappointed", "poor", "weak", "stupid", "dumb", "ridiculous",
    "unfunny", "mess", "waste",
    # direct emotion
    "happy", "sad", "angry", "upset", "excited", "depressed",
    "frustrated", "surprised", "shocked", "scared", "afraid",
    "emotional"
}

In [42]:
# 데이터 샘플 확인
data_path = r"C:\Users\wanje\Desktop\deeplearning\Raw\mosi_text_metadata.csv"
df = pd.read_csv(data_path)

print(f"데이터 수: {len(df)}")
print(f"\n샘플 확인:")
print(df[["text"]].head(5))

데이터 수: 2199

샘플 확인:
                                                text
0                          ANYHOW IT WAS REALLY GOOD
1  THAY DID THEY DIDNT REALLY DO A WHOLE BUNCH OF...
2                 I MEAN THEY DID A LITTLE BIT OF IT
3                              BUT NOT A WHOLE BUNCH
4                          AND THEY SHOULDVE I GUESS


데이터 전처리 <소문자 변환 / 토큰화 / 불용어 제거>

In [43]:
# 텍ㅅ트에 데이터 전처리 적용 => 1차시 40p
def preprocess_text(text, remove_emotion_words=False):
    text = text.lower() # 소문자 변환
    text = re.sub(r'[^\w\s]', '', text) # 특수문자 제거
    tokens = text.split() # 토큰화 
    tokens = [token for token in tokens if token not in stop_words] # 불용어 제거
    
    # 감정 단어 제거
    if remove_emotion_words:
        tokens = [tok for tok in tokens if tok not in emotion_words]

    return tokens

# 일반 전처리 
df["tokens"] = df["text"].apply(lambda x: preprocess_text(x, False))

# 감정 단어 제거 전처리
df["tokens_no_emotion"] = df["text"].apply(lambda x: preprocess_text(x, True))

print("전처리 전후 비교:")
for i in range(3):
    print(f"  원본:  {df['text'].iloc[i]}")
    print(f"  결과:  {df['tokens'].iloc[i]}")
    print(f"감정 제거: {df['tokens_no_emotion'].iloc[i]}")
    print()

전처리 전후 비교:
  원본:  ANYHOW IT WAS REALLY GOOD
  결과:  ['anyhow', 'really', 'good']
감정 제거: ['anyhow', 'really']

  원본:  THAY DID THEY DIDNT REALLY DO A WHOLE BUNCH OF BACKGROUND INFO ON WHY SHE HAS TO FIGHT AND BE PREPARED
  결과:  ['thay', 'didnt', 'really', 'whole', 'bunch', 'background', 'info', 'why', 'fight', 'prepared']
감정 제거: ['thay', 'didnt', 'really', 'whole', 'bunch', 'background', 'info', 'why', 'fight', 'prepared']

  원본:  I MEAN THEY DID A LITTLE BIT OF IT
  결과:  ['mean', 'little', 'bit']
감정 제거: ['mean', 'little', 'bit']



정수 인코딩: 단어 사전 구축 / 정수 인코딩

In [44]:
#정수 인코딩 => 데이터에 등장하는 단어를 모아 각ㄱ 단어에 고유 번호 붙이기
all_tokens = []
for tokens in df["tokens"]:
    for token in tokens:
        all_tokens.append(token)

word_counts = Counter(all_tokens)
print(f"전체 고유 단어 수: {len(word_counts)}")
print(f"가장 많이 등장한 단어 10개: {word_counts.most_common(10)}")

MIN_FREQ = 2
vocab = {"<PAD>": 0, "<UNK>": 1}
idx = 2

for word, count in word_counts.items():
    if count >= MIN_FREQ:
        vocab[word] = idx
        idx += 1

print(f"Vocab 크기: {len(vocab)}")


전체 고유 단어 수: 3036
가장 많이 등장한 단어 10개: [('like', 433), ('um', 421), ('movie', 407), ('really', 369), ('but', 354), ('just', 280), ('so', 204), ('know', 203), ('not', 169), ('good', 157)]
Vocab 크기: 1368


In [45]:
# 감정 단어 제거 버전 정수 인코딩
all_tokens_no_emotion = []
for tokens in df["tokens_no_emotion"]:
    for token in tokens:
        all_tokens_no_emotion.append(token)

word_counts_no_emotion = Counter(all_tokens_no_emotion)
print(f"전체 고유 단어 수 (감정 단어 제거): {len(word_counts_no_emotion)}")
print(f"가장 많이 등장한 단어 10개 (감정 단어 제거): {word_counts_no_emotion.most_common(10)}")

vocab_no_emotion = {"<PAD>": 0, "<UNK>": 1}
idx = 2
for word, count in word_counts_no_emotion.items():
    if count >= MIN_FREQ:
        vocab_no_emotion[word] = idx
        idx += 1
print(f"Vocab 크기 (감정 단어 제거): {len(vocab_no_emotion)}")

전체 고유 단어 수 (감정 단어 제거): 2981
가장 많이 등장한 단어 10개 (감정 단어 제거): [('um', 421), ('movie', 407), ('really', 369), ('but', 354), ('just', 280), ('so', 204), ('know', 203), ('not', 169), ('think', 128), ('one', 124)]
Vocab 크기 (감정 단어 제거): 1320


In [46]:
# 정수 인코딩 함수
def encode_tokens(tokens, vocab):
    encoded = []
    for word in tokens:
        if word in vocab:
            encoded.append(vocab[word])
        else:
            encoded.append(vocab["<UNK>"])
    return encoded

# 일반 정수 인코딩
encoded_list = []
for t in df["tokens"]:
    encoded_list.append(encode_tokens(t, vocab))
df["encoded"] = encoded_list

# 감정 단어 제거 버전 정수 인코딩
encoded_list_no_emotion = []
for t in df["tokens_no_emotion"]:
    encoded_list_no_emotion.append(encode_tokens(t, vocab_no_emotion))
df["encoded_no_emotion"] = encoded_list_no_emotion

In [47]:
# 일반 인코딩 결과 저장
save_dir = r"C:\Users\wanje\Desktop\deeplearning"

save_data = {
    "encoded": df["encoded"].tolist(),
    "labels": df["label"].values,
    "video_ids": df["video_id"].values,
    "vocab": vocab,
    "texts": df["text"].values,
}

save_path = os.path.join(save_dir, "text_preprocessed_data.pkl")
with open(save_path, "wb") as f:
    pickle.dump(save_data, f)

print(f"저장 완료: {save_path}")
print(f"  데이터 수: {len(save_data['encoded'])}")
print(f"  Vocab 크기: {len(vocab)}")

저장 완료: C:\Users\wanje\Desktop\deeplearning\text_preprocessed_data.pkl
  데이터 수: 2199
  Vocab 크기: 1368


In [48]:
# 감정 제거 인코딩 결과 저장
save_data_no_emotion = {
    "encoded": df["encoded_no_emotion"].tolist(),
    "labels": df["label"].values,
    "video_ids": df["video_id"].values,
    "vocab": vocab_no_emotion,
    "texts": df["text"].values,
}
save_path_no_emotion = os.path.join(save_dir, "text_preprocessed_data_no_emotion.pkl")
with open(save_path_no_emotion, "wb") as f:
    pickle.dump(save_data_no_emotion, f)

print(f"저장 완료: {save_path_no_emotion}")
print(f"  데이터 수: {len(save_data_no_emotion['encoded'])}")
print(f"  Vocab 크기: {len(vocab_no_emotion)}")

저장 완료: C:\Users\wanje\Desktop\deeplearning\text_preprocessed_data_no_emotion.pkl
  데이터 수: 2199
  Vocab 크기: 1320


2차시 => 패딩(길이통일) + 임베딩(숫자->벡터)

In [49]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer, AutoModel

In [55]:
save_dir1 = r"C:\Users\wanje\Desktop\deeplearning\text_preprocessed_data.pkl"
with open(save_dir1, "rb") as last_file:
    text_data1 = pickle.load(last_file)

# 가장 긴 문장 길이 패딩
max_len = 0
for seq in text_data1["encoded"]:
    if len(seq) > max_len:
        max_len = len(seq)

padded_max = []
for seq in text_data1["encoded"]:
    pad_count = max_len - len(seq)
    padded_max.append(seq + [vocab["<PAD>"]] * pad_count)

fixed_len = 10
padded_fixed1 = []
for seq in text_data1["encoded"]:
    fixed_seq = seq[:fixed_len]  # 고정 길이로 자르기
    pad_count = fixed_len - len(fixed_seq)  # 패딩할 개수
    padded_fixed1.append(fixed_seq + [vocab["<PAD>"]] * pad_count)


text_data1["padded_fixed1"] = padded_fixed1

print("0번째 문장:", text_data1["texts"][0])
print("정수 인코딩 결과:", text_data1["encoded"][0])
print("고정 길이 패딩:", text_data1["padded_fixed1"][0])

0번째 문장: ANYHOW IT WAS REALLY GOOD
정수 인코딩 결과: [1, 2, 3]
고정 길이 패딩: [1, 2, 3, 0, 0, 0, 0, 0, 0, 0]


In [56]:
save_dir2 = r"C:\Users\wanje\Desktop\deeplearning\text_preprocessed_data_no_emotion.pkl"
with open(save_dir2, "rb") as last_file:
    text_data2 = pickle.load(last_file)

# 가장 긴 문장 길이 패딩
max_len = 0
for seq in text_data2["encoded"]:
    if len(seq) > max_len:
        max_len = len(seq)

padded_max = []
for seq in text_data2["encoded"]:
    pad_count = max_len - len(seq)
    padded_max.append(seq + [vocab["<PAD>"]] * pad_count)

fixed_len = 10
padded_fixed2 = []
for seq in text_data2["encoded"]:
    fixed_seq = seq[:fixed_len]  # 고정 길이로 자르기
    pad_count = fixed_len - len(fixed_seq)  # 패딩할 개수
    padded_fixed2.append(fixed_seq + [vocab["<PAD>"]] * pad_count)
text_data2["padded_fixed2"] = padded_fixed2

print("0번째 문장:", text_data2["texts"][0])
print("정수 인코딩 결과:", text_data2["encoded"][0])
print("고정 길이 패딩:", text_data2["padded_fixed2"][0])

0번째 문장: ANYHOW IT WAS REALLY GOOD
정수 인코딩 결과: [1, 2]
고정 길이 패딩: [1, 2, 0, 0, 0, 0, 0, 0, 0, 0]


In [57]:
embedding_dim = 4

embedding_table = torch.randn(len(vocab), embedding_dim)
embedding_table[vocab["<PAD>"]] = 0

print("임베딩 테이블 shape:", embedding_table.shape)
print("PAD 벡터:", embedding_table[vocab["<PAD>"]])

임베딩 테이블 shape: torch.Size([1368, 4])
PAD 벡터: tensor([0., 0., 0., 0.])


In [66]:
# 임베딩: 단어의 의미를 벡터로 표현하기 위한 과정
# Embedding 구현 
sample_seq1 = padded_fixed1[0] 

sentence_vectors = []

for token_idx in sample_seq1:
    token_vector = embedding_table[token_idx]
    sentence_vectors.append(token_vector)

sentence_vectors = torch.stack(sentence_vectors)

print("0번째 문장:", text_data1["texts"][0])
print("0번째 문장 정수 인코딩:", sample_seq1)
print("문장 길이:", len(sample_seq1))
print("\n0번째 문장의 임베딩 결과 shape:", sentence_vectors.shape)
print("0번째 문장의 임베딩 결과:", (sentence_vectors))


# nn.Embedding 활용
X = torch.tensor(padded_fixed1, dtype=torch.long)
y = torch.tensor(text_data1["labels"], dtype=torch.long)

embedding_layer = nn.Embedding(
    num_embeddings=len(vocab),
    embedding_dim=4,
    padding_idx=vocab["<PAD>"]
)

embedded_text = embedding_layer(X)
print("\nnn.Embedding 결과 shape:", embedded_text.shape)
print("첫 번째 문장 임베딩 결과:")
print(embedded_text[0])

0번째 문장: ANYHOW IT WAS REALLY GOOD
0번째 문장 정수 인코딩: [1, 2, 3, 0, 0, 0, 0, 0, 0, 0]
문장 길이: 10

0번째 문장의 임베딩 결과 shape: torch.Size([10, 4])
0번째 문장의 임베딩 결과: tensor([[-0.4101,  0.7746,  0.3254, -0.5968],
        [-0.5627,  1.8853, -0.4569,  1.3746],
        [ 1.1402, -1.1516, -0.5224, -1.4374],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000]])

nn.Embedding 결과 shape: torch.Size([2199, 10, 4])
첫 번째 문장 임베딩 결과:
tensor([[-0.4915, -1.1115,  0.1734, -0.8648],
        [-0.3897, -1.2307, -0.3672, -0.4115],
        [-0.8167, -0.2783,  1.1394,  0.7526],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
  

In [67]:
# 임베딩: 단어의 의미를 벡터로 표현하기 위한 과정
sample_seq2 = padded_fixed2[0] 

sentence_vectors = []

for token_idx in sample_seq2:
    token_vector = embedding_table[token_idx]
    sentence_vectors.append(token_vector)

sentence_vectors = torch.stack(sentence_vectors)

print("0번째 문장:", text_data2["texts"][0])
print("0번째 문장 정수 인코딩:", sample_seq2)
print("문장 길이:", len(sample_seq2))
print("\n0번째 문장의 임베딩 결과 shape:", sentence_vectors.shape)
print("0번째 문장의 임베딩 결과:", (sentence_vectors))

# nn.Embedding 활용
X = torch.tensor(padded_fixed2, dtype=torch.long)
y = torch.tensor(text_data2["labels"], dtype=torch.long)

embedding_layer = nn.Embedding(
    num_embeddings=len(vocab),
    embedding_dim=4,
    padding_idx=vocab["<PAD>"]
)

embedded_text = embedding_layer(X)
print("\nnn.Embedding 결과 shape:", embedded_text.shape)
print("첫 번째 문장 임베딩 결과:")
print(embedded_text[0])

0번째 문장: ANYHOW IT WAS REALLY GOOD
0번째 문장 정수 인코딩: [1, 2, 0, 0, 0, 0, 0, 0, 0, 0]
문장 길이: 10

0번째 문장의 임베딩 결과 shape: torch.Size([10, 4])
0번째 문장의 임베딩 결과: tensor([[-0.4101,  0.7746,  0.3254, -0.5968],
        [-0.5627,  1.8853, -0.4569,  1.3746],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000]])

nn.Embedding 결과 shape: torch.Size([2199, 10, 4])
첫 번째 문장 임베딩 결과:
tensor([[-0.2053,  0.0572, -0.9496,  0.8073],
        [ 0.8261, -0.0718, -1.3698, -0.5583],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
  

In [68]:
# 사전학습 모델 활용
model_name = "google-bert/bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

sentence = text_data1["texts"][0]

inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
embeddings = outputs.last_hidden_state[0]

print("문장:", sentence)

for token, emb in zip(tokens, embeddings):
    print(f"토큰: {token}")
    print(f"벡터 앞 5개 값: {emb[:5].tolist()}")
    print()

문장: ANYHOW IT WAS REALLY GOOD
토큰: [CLS]
벡터 앞 5개 값: [0.27537861466407776, 0.5214540958404541, 0.15492990612983704, -0.03061075508594513, -0.21965113282203674]

토큰: AN
벡터 앞 5개 값: [-1.043573021888733, -0.1528216451406479, 0.19757944345474243, 0.7298051118850708, 0.1045314371585846]

토큰: ##Y
벡터 앞 5개 값: [-0.3096957802772522, -0.12426143139600754, -0.18246303498744965, 0.6591668128967285, 0.26444560289382935]

토큰: ##H
벡터 앞 5개 값: [-0.4722752273082733, 0.995591402053833, -0.027917858213186264, 0.7027592658996582, -0.22906605899333954]

토큰: ##OW
벡터 앞 5개 값: [0.13739891350269318, 0.23970404267311096, -0.14093263447284698, 0.1297909915447235, 0.26485317945480347]

토큰: IT
벡터 앞 5개 값: [-0.3721908628940582, 0.19955213367938995, 0.02042205259203911, 0.9711361527442932, 0.15533792972564697]

토큰: WA
벡터 앞 5개 값: [-0.8836513161659241, 0.33360767364501953, -0.0933530256152153, 0.9428226947784424, 0.11695477366447449]

토큰: ##S
벡터 앞 5개 값: [-0.19380132853984833, 0.23485338687896729, 0.38944777846336365, 0.83773

In [69]:
# 사전학습 모델 활용
model_name = "google-bert/bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

sentence = text_data2["texts"][0]

inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
embeddings = outputs.last_hidden_state[0]

print("문장:", sentence)

for token, emb in zip(tokens, embeddings):
    print(f"토큰: {token}")
    print(f"벡터 앞 5개 값: {emb[:5].tolist()}")
    print()

문장: ANYHOW IT WAS REALLY GOOD
토큰: [CLS]
벡터 앞 5개 값: [0.27537861466407776, 0.5214540958404541, 0.15492990612983704, -0.03061075508594513, -0.21965113282203674]

토큰: AN
벡터 앞 5개 값: [-1.043573021888733, -0.1528216451406479, 0.19757944345474243, 0.7298051118850708, 0.1045314371585846]

토큰: ##Y
벡터 앞 5개 값: [-0.3096957802772522, -0.12426143139600754, -0.18246303498744965, 0.6591668128967285, 0.26444560289382935]

토큰: ##H
벡터 앞 5개 값: [-0.4722752273082733, 0.995591402053833, -0.027917858213186264, 0.7027592658996582, -0.22906605899333954]

토큰: ##OW
벡터 앞 5개 값: [0.13739891350269318, 0.23970404267311096, -0.14093263447284698, 0.1297909915447235, 0.26485317945480347]

토큰: IT
벡터 앞 5개 값: [-0.3721908628940582, 0.19955213367938995, 0.02042205259203911, 0.9711361527442932, 0.15533792972564697]

토큰: WA
벡터 앞 5개 값: [-0.8836513161659241, 0.33360767364501953, -0.0933530256152153, 0.9428226947784424, 0.11695477366447449]

토큰: ##S
벡터 앞 5개 값: [-0.19380132853984833, 0.23485338687896729, 0.38944777846336365, 0.83773